# 160 — Weight Matrix Probing

Analyze weight matrices for interpretable structure: singular value spectra,
low-rank patterns, QK/OV alignment, and embedding-unembedding correspondence.

## Why This Matters

Probing weight matrix tests what information is linearly accessible in model representations. Linear probes provide a principled way to measure whether specific concepts (syntax, semantics, factual knowledge) are encoded at each layer.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_matrix_probing import (
    weight_spectrum,
    weight_alignment,
    weight_norms_profile,
    low_rank_structure,
    embed_unembed_alignment,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

In [ ]:
result = weight_spectrum(model, layer=0, component='attn')
for name, info in result['matrices'].items():
    print(f"{name} {info['shape']}: eff_rank={info['effective_rank']:.1f}  "
          f"cond={info['condition_number']:.0f}  top1={info['top1_variance_fraction']:.3f}  "
          f"top5={info['top5_variance_fraction']:.3f}")

In [ ]:
result = weight_alignment(model, layer=0)
for h in result['per_head']:
    print(f"Head {h['head']}: QK_norm={h['qk_norm']:.4f}  OV_norm={h['ov_norm']:.4f}  "
          f"OV_rank={h['ov_effective_rank']:.1f}")

In [ ]:
result = weight_norms_profile(model)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: Q={p['W_Q_norm']:.3f} K={p['W_K_norm']:.3f} V={p['W_V_norm']:.3f} "
          f"O={p['W_O_norm']:.3f} in={p['W_in_norm']:.3f} out={p['W_out_norm']:.3f}")

In [ ]:
result = low_rank_structure(model, layer=0, rank_threshold=0.95)
print(f"Low-rank matrices: {result['n_low_rank']}\n")
for name, info in result['matrices'].items():
    tag = ' LOW-RANK' if info['is_low_rank'] else ''
    print(f"{name:5s}: rank_needed={info['rank_for_threshold']}/{info['full_rank']}  "
          f"compression={info['compression_ratio']:.3f}{tag}")

In [ ]:
result = embed_unembed_alignment(model, top_k=5)
print(f"Mean alignment: {result['mean_alignment']:.4f} +/- {result['std_alignment']:.4f}\n")
print("Most aligned:")
for t in result['most_aligned']:
    print(f"  token {t['token']}: cos={t['cosine']:.4f}")
print("Least aligned:")
for t in result['least_aligned']:
    print(f"  token {t['token']}: cos={t['cosine']:.4f}")